<a href="https://colab.research.google.com/github/SreePavani1103/24P31A05D5_DriveReady_OPC_2026/blob/main/Day4/task4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q -U langchain langchain-groq langgraph gspread pydantic

In [4]:
import os
import gspread
from google.colab import auth, userdata
from google.auth import default
from langchain_groq import ChatGroq

# 1. Load GROQ_API_KEY from Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# 2. Authenticate with Google for Sheets / Drive access
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

SHEET_TITLE = "Sweet Crumbs Orders"
HEADERS = ["name", "item", "qty", "date", "eggless", "total"]


def get_or_create_orders_sheet(client, title=SHEET_TITLE, headers=HEADERS):
  """Open an existing order sheet or create it once with headers."""
  try:
    spreadsheet = client.open(title)
    worksheet = spreadsheet.sheet1
  except gspread.SpreadsheetNotFound:
    spreadsheet = client.create(title)
    worksheet = spreadsheet.sheet1
    worksheet.append_row(headers)

  # Ensure headers exist if sheet1 is completely empty
  existing_values = worksheet.get_all_values()
  if not existing_values:
    worksheet.append_row(headers)

  return spreadsheet, worksheet


sh, ws = get_or_create_orders_sheet(gc)
print(f"Connected to sheet: {sh.title} ({sh.url})")

# 3. Initialize Groq chat model
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
)

Connected to sheet: Sweet Crumbs Orders (https://docs.google.com/spreadsheets/d/12yW1p-_op51uHBqA4GCi5fdn4KUSjyJPcunEFbPtfkc)


In [5]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}
HOURS = {"monday": "9 am to 8 pm", "tuesday": "9 am to 8 pm", "wednesday": "9 am to 8 pm",
         "thursday": "9 am to 8 pm", "friday": "9 am to 8 pm", "saturday": "9 am to 8 pm",
         "sunday": "10 am to 6 pm"}

class Order(BaseModel):
    """A cake order taken from a customer's chat message."""
    name: str = Field(description="Customer's first name")
    item: Literal["plum cake", "chocolate cake", "cupcake"] = Field(description="Menu item")
    qty: int = Field(description="How many", ge=1)
    date: str = Field(description="Pickup date as the customer said it")
    eggless: bool = Field(description="True if the customer asked for eggless")

@tool
def get_price(item: str, qty: int = 1) -> str:
    """Get the total price in rupees for a quantity of one menu item.

    Args:
        item: Menu item, e.g. "plum cake", "chocolate cake" or "cupcake"
        qty: How many the customer wants
    """
    price = MENU.get(item.lower().strip())
    if price is None:
        return f"'{item}' is not on the menu. We have: {', '.join(MENU)}."
    return f"{qty} x {item} = Rs {price * qty}"

@tool
def is_open(day: str) -> str:
    """Check the bakery's opening hours for a day of the week, e.g. "sunday"."""
    hours = HOURS.get(day.lower().strip())
    return f"Open {hours} on {day}." if hours else f"'{day}' is not a day of the week."

@tool
def log_order(name: str, item: str, qty: int, date: str, eggless: bool) -> str:
    """Save a confirmed cake order to the Orders sheet.
    Use only when the customer has given their name, item, quantity and pickup date."""
    try:
        order = Order(name=name, item=item.lower().strip(), qty=qty, date=date, eggless=eggless)
    except Exception as e:
        return f"Could not save the order: {e}. Ask the customer to correct it."
    total = MENU[order.item] * order.qty
    ws.append_row([order.name, order.item, order.qty, order.date,
                   "yes" if order.eggless else "no", total])
    return f"Saved: {order.qty} x {order.item} for {order.name} on {order.date}, Rs {total}."

In [6]:
from langchain.agents import create_agent

SYSTEM = """ROLE:   You are the friendly order assistant for Sweet Crumbs Home Bakery.
FACTS:  Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Always use get_price for prices and is_open for timings; never guess them.
        Before log_order, make sure you have name, item, quantity and pickup date; ask if missing.
        After saving, read the order back with the total.
FORMAT: Under 60 words, friendly, English or Telugu matching the customer."""

agent = create_agent(model=llm, tools=[get_price, is_open, log_order], system_prompt=SYSTEM)

history = []
def chat(text: str) -> str:
    global history
    try:
        result = agent.invoke({"messages": history + [{"role": "user", "content": text}]})
        history = result["messages"]                  # keep the whole run as memory
        return history[-1].content
    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I'm a little busy. Please try again in a minute."

print(chat("Are you open on Sunday? How much are 2 chocolate cakes?"))
print(chat("I'm Ravi. Please book 2 eggless chocolate cakes for 24 Dec."))   # check your s

Agent error: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}
Sorry, I'm a little busy. Please try again in a minute.
Agent error: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}
Sorry, I'm a little busy. Please try again in a minute.


In [7]:
for m in history:
    m.pretty_print()